In [2]:
import math
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from lightgbm import LGBMRanker
except ImportError as e:
    raise ImportError("Please install lightgbm: pip install lightgbm") from e

pd.set_option("display.max_columns", 200)
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "Train.csv"
TEST_PATH = DATA_DIR / "Test.csv"
SKILLS_PATH = DATA_DIR / "Skills.csv"
OCCUPATIONS_PATH = DATA_DIR / "Occupations.csv"
SUBMISSION_PATH = DATA_DIR / "submission.csv"

CFG = {
    "top_k_candidates": 100,
    "lgbm_n_estimators": 350,
    "lgbm_learning_rate": 0.05,
    "lgbm_num_leaves": 63,
    "lgbm_min_child_samples": 30,
}

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
skills = pd.read_csv(SKILLS_PATH)
occupations = pd.read_csv(OCCUPATIONS_PATH)


def norm_id(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip()
    if s.endswith(".0") and s.replace(".", "", 1).isdigit():
        s = s[:-2]
    return s


train["ID"] = train["ID"].map(norm_id)
test["ID"] = test["ID"].map(norm_id)
skills["ID"] = skills["ID"].map(norm_id)
occupations["ID"] = occupations["ID"].map(norm_id)

skill_cols = ["skill_1", "skill_2", "skill_3", "skill_4", "skill_5"]
occ_cols = ["occ_1", "occ_2", "occ_3", "occ_4", "occ_5"]

for c in skill_cols + occ_cols:
    train[c] = train[c].map(norm_id)
for c in skill_cols:
    test[c] = test[c].map(norm_id)

print("train:", train.shape)
print("test:", test.shape)
print("skills:", skills.shape)
print("occupations:", occupations.shape)
train.head(2)

train: (2790, 11)
test: (1196, 6)
skills: (3145, 8)
occupations: (977, 14)


,ID,skill_1,skill_2,skill_3,skill_4,skill_5,occ_1,occ_2,occ_3,occ_4,occ_5
0,5IZ31YRUJOQW,KS124986R0H6NGX3SQP3,KS7G747655VG23WXMS9B,KS122VT6S2JJ5C5D80NF,KS1218W78FGVPVP2KXPX,KS441LY691MT0N689MWR,23131115,17141810,23131110,23131116,17111410
1,81N0GJOF9USK,KS125TB6YR6236RKM563,KS120VB76P5WB69FVNRT,KS1224W5XSXCJBNSLLVX,KS122C06Z8NDM5G6NYNT,ESE8E6C89348587B4CCC,23171710,27111320,27111318,23171521,23171524


In [4]:
# Prepare metadata encoders
occ_meta = occupations.drop_duplicates("ID").set_index("ID")
occ_feature_cols = [c for c in occupations.columns if c != "ID"]

for c in occ_feature_cols:
    occ_meta[c] = occ_meta[c].astype(str).fillna("unknown")


def fit_label_map(values):
    vals = sorted(set(values))
    return {v: i for i, v in enumerate(vals)}


occ_label_maps = {c: fit_label_map(occ_meta[c].tolist() + ["unknown"]) for c in occ_feature_cols}

all_skill_values = set(pd.unique(train[skill_cols].values.ravel()).tolist() + pd.unique(test[skill_cols].values.ravel()).tolist())
all_skill_values = {str(s) for s in all_skill_values if pd.notna(s)}
all_skill_values.add("unknown")
skill_label_map = fit_label_map(all_skill_values)


def encode_skill_slots(skills_in, k=5):
    slots = list(skills_in[:k]) + ["unknown"] * max(0, k - len(skills_in))
    return [skill_label_map.get(str(s), skill_label_map["unknown"]) for s in slots[:k]]


def get_occ_feature(occ, col):
    if occ in occ_meta.index:
        v = str(occ_meta.at[occ, col])
    else:
        v = "unknown"
    return occ_label_maps[col].get(v, occ_label_maps[col]["unknown"])

print("occupation metadata cols:", len(occ_feature_cols))
print("encoded skill vocab:", len(skill_label_map))

occupation metadata cols: 13
encoded skill vocab: 3162


## Preprocessing, Training, and Testing

This section runs the full modeling pipeline in one place.

Preprocessing:
- Split `Train.csv` at the query level into a fit subset and a local validation subset.
- Build candidate-retrieval statistics only from the fit subset to avoid leaking validation labels into candidate generation.
- Use `skills.csv` and `occupations.csv` encodings prepared above as lightweight metadata features for the reranker.

Training and validation:
- Generate top candidates from exact 5-skill matches, skill-pair matches, single-skill co-occurrence, and a weak prior.
- Train an `LGBMRanker` on the fit subset.
- Score the held-out queries and report a local CV MAP@5.

Testing:
- Rebuild retrieval statistics on the full training set.
- Refit the ranker on all training queries.
- Predict the top-5 occupations for each row in `test.csv`.

In [5]:
drop_cols = ["qid", "label", "candidate_occ"]

all_row_idx = np.arange(len(train))
rng = np.random.default_rng(SEED)
rng.shuffle(all_row_idx)

val_frac = 0.20
n_val_rows = max(1, int(len(all_row_idx) * val_frac))
val_idx = all_row_idx[:n_val_rows]
fit_idx = all_row_idx[n_val_rows:]

train_fit = train.iloc[fit_idx].reset_index(drop=True)
train_val = train.iloc[val_idx].reset_index(drop=True)


def build_artifacts(source_df):
    skill_occ = defaultdict(Counter)
    pair_occ = defaultdict(Counter)
    combo_occ = defaultdict(Counter)
    occ_counter = Counter()
    occ_skills = defaultdict(set)

    for row in source_df.itertuples(index=False):
        qskills = sorted({getattr(row, c) for c in skill_cols if getattr(row, c) != "unknown"})
        true_occs = [getattr(row, c) for c in occ_cols if getattr(row, c) != "unknown"]

        if not qskills or not true_occs:
            continue

        for occ in true_occs:
            combo_occ[tuple(qskills)][occ] += 1

        for i, s1 in enumerate(qskills):
            for s2 in qskills[i + 1 :]:
                for occ in true_occs:
                    pair_occ[(s1, s2)][occ] += 1

        for skill in qskills:
            for occ in true_occs:
                skill_occ[skill][occ] += 1
                occ_counter[occ] += 1
                occ_skills[occ].add(skill)

    total_pairs = max(sum(sum(v.values()) for v in skill_occ.values()), 1)
    skill_df = {skill: len(cnts) for skill, cnts in skill_occ.items()}
    skill_weights = {
        skill: math.log((1 + total_pairs) / (1 + df)) + 1.0
        for skill, df in skill_df.items()
    }
    occ_prior_local = {
        occ: cnt / max(sum(occ_counter.values()), 1)
        for occ, cnt in occ_counter.items()
    }
    global_rank = [occ for occ, _ in occ_counter.most_common()]

    return {
        "skill_occ": skill_occ,
        "pair_occ": pair_occ,
        "combo_occ": combo_occ,
        "occ_skills": occ_skills,
        "skill_weights": skill_weights,
        "occ_prior": occ_prior_local,
        "global_rank": global_rank,
    }


def get_candidates(qskills, artifacts, top_k):
    qskills = sorted({skill for skill in qskills if skill != "unknown"})
    scores = defaultdict(float)

    combo_hits = artifacts["combo_occ"].get(tuple(qskills))
    if combo_hits:
        for occ, cnt in combo_hits.items():
            scores[occ] += 20.0 + 2.0 * math.log(1.0 + cnt)

    for i, s1 in enumerate(qskills):
        for s2 in qskills[i + 1 :]:
            for occ, cnt in artifacts["pair_occ"].get((s1, s2), {}).items():
                scores[occ] += 1.25 * math.log(1.0 + cnt)

    for skill in qskills:
        occ_hits = artifacts["skill_occ"].get(skill)
        if not occ_hits:
            continue
        total = sum(occ_hits.values())
        weight = artifacts["skill_weights"].get(skill, 1.0)
        for occ, cnt in occ_hits.items():
            local_rate = (cnt + 1.0) / (total + 1.0)
            scores[occ] += weight * math.log(1.0 + cnt) + 0.3 * math.log(local_rate + 1e-12)

    for occ, p in artifacts["occ_prior"].items():
        scores[occ] += 0.03 * math.log(max(p, 1e-12))

    ranked = [occ for occ, _ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)]
    if len(ranked) < top_k:
        seen = set(ranked)
        ranked.extend([occ for occ in artifacts["global_rank"] if occ not in seen])

    return ranked[:top_k], scores


def build_ranker_df(source_df, artifacts, start_qid):
    rows = []
    group_sizes = []

    for qid, row in enumerate(source_df.itertuples(index=False), start=start_qid):
        qskills = [str(getattr(row, c)) for c in skill_cols]
        true_set = {str(getattr(row, c)) for c in occ_cols if str(getattr(row, c)) != "unknown"}
        candidates, cand_scores = get_candidates(qskills, artifacts, CFG["top_k_candidates"])
        qskill_set = set(qskills)
        skill_slots = encode_skill_slots(qskills, 5)
        group_start = len(rows)

        for rank_idx, occ in enumerate(candidates, start=1):
            overlap_count = len(qskill_set.intersection(artifacts["occ_skills"].get(occ, set())))
            item = {
                "qid": qid,
                "label": int(occ in true_set),
                "candidate_occ": occ,
                "cand_rank": rank_idx,
                "cand_score": float(cand_scores.get(occ, -50.0)),
                "occ_prior_log": math.log(max(artifacts["occ_prior"].get(occ, 1e-12), 1e-12)),
                "overlap_count": overlap_count,
                "overlap_ratio": overlap_count / max(len(qskill_set), 1),
                "query_skill_count": len(qskill_set),
            }

            for i in range(5):
                item[f"skill_{i+1}_enc"] = skill_slots[i]

            for c in occ_feature_cols:
                item[f"occ_{c.lower()}_enc"] = get_occ_feature(occ, c)

            rows.append(item)

        group_sizes.append(len(rows) - group_start)

    return pd.DataFrame(rows), np.asarray(group_sizes, dtype=np.int32)


def apk(actual, predicted, k=5):
    if not actual:
        return 0.0
    score = 0.0
    hits = 0.0
    predicted = predicted[:k]
    for i, occ in enumerate(predicted, start=1):
        if occ in actual and occ not in predicted[: i - 1]:
            hits += 1.0
            score += hits / i
    return score / min(len(actual), k)


fit_artifacts = build_artifacts(train_fit)
train_rank_df, group_train = build_ranker_df(train_fit, fit_artifacts, start_qid=1)
val_rank_df, group_val = build_ranker_df(train_val, fit_artifacts, start_qid=len(train_fit) + 1)

feature_cols = [c for c in train_rank_df.columns if c not in drop_cols]
reranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=CFG["lgbm_n_estimators"],
    learning_rate=CFG["lgbm_learning_rate"],
    num_leaves=CFG["lgbm_num_leaves"],
    min_child_samples=CFG["lgbm_min_child_samples"],
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=SEED,
    force_row_wise=True,
    verbosity=-1,
)

reranker.fit(
    train_rank_df[feature_cols],
    train_rank_df["label"].astype(int),
    group=group_train,
 )

val_rank_df = val_rank_df.copy()
val_rank_df["pred_score"] = reranker.predict(val_rank_df[feature_cols])
cv_scores = []
for _, group_df in val_rank_df.groupby("qid", sort=False):
    actual = group_df.loc[group_df["label"] == 1, "candidate_occ"].tolist()
    predicted = group_df.sort_values("pred_score", ascending=False)["candidate_occ"].head(5).tolist()
    cv_scores.append(apk(actual, predicted, k=5))

local_cv_map5 = float(np.mean(cv_scores)) if cv_scores else 0.0
print("fit queries:", len(group_train), "| val queries:", len(group_val))
print(f"local CV MAP@5: {local_cv_map5:.6f}")

full_artifacts = build_artifacts(train)
ranker_train_df, full_group = build_ranker_df(train, full_artifacts, start_qid=1)
feature_cols = [c for c in ranker_train_df.columns if c not in drop_cols]
reranker.fit(
    ranker_train_df[feature_cols],
    ranker_train_df["label"].astype(int),
    group=full_group,
 )
print("LGBMRanker refit on full training data with", len(feature_cols), "features")

skill_occ_counts = full_artifacts["skill_occ"]
pair_occ_counts = full_artifacts["pair_occ"]
combo_occ_counts = full_artifacts["combo_occ"]
occ_skill_set = full_artifacts["occ_skills"]
skill_weight = full_artifacts["skill_weights"]
occ_prior = full_artifacts["occ_prior"]
global_occ_rank = full_artifacts["global_rank"]

submission_rows = []
for row in test.itertuples(index=False):
    qskills = [str(getattr(row, c)) for c in skill_cols]
    candidates, cand_scores = get_candidates(qskills, full_artifacts, CFG["top_k_candidates"])
    qskill_set = set(qskills)
    skill_slots = encode_skill_slots(qskills, 5)
    pred_rows = []

    for rank_idx, occ in enumerate(candidates, start=1):
        overlap_count = len(qskill_set.intersection(full_artifacts["occ_skills"].get(occ, set())))
        item = {
            "candidate_occ": occ,
            "cand_rank": rank_idx,
            "cand_score": float(cand_scores.get(occ, -50.0)),
            "occ_prior_log": math.log(max(full_artifacts["occ_prior"].get(occ, 1e-12), 1e-12)),
            "overlap_count": overlap_count,
            "overlap_ratio": overlap_count / max(len(qskill_set), 1),
            "query_skill_count": len(qskill_set),
        }

        for i in range(5):
            item[f"skill_{i+1}_enc"] = skill_slots[i]

        for c in occ_feature_cols:
            item[f"occ_{c.lower()}_enc"] = get_occ_feature(occ, c)

        pred_rows.append(item)

    pred_df = pd.DataFrame(pred_rows)
    pred_df["score"] = reranker.predict(pred_df[feature_cols])
    top5 = pred_df.sort_values("score", ascending=False)["candidate_occ"].head(5).tolist()

    if len(top5) < 5:
        seen = set(top5)
        top5.extend([occ for occ in full_artifacts["global_rank"] if occ not in seen][: 5 - len(top5)])

    submission_rows.append({
        "ID": str(row.ID),
        "occ_1": top5[0],
        "occ_2": top5[1],
        "occ_3": top5[2],
        "occ_4": top5[3],
        "occ_5": top5[4],
    })

submission = pd.DataFrame(submission_rows)
submission = submission[["ID", "occ_1", "occ_2", "occ_3", "occ_4", "occ_5"]]
submission.head()

fit queries: 2232 | val queries: 558
local CV MAP@5: 0.256909
LGBMRanker refit on full training data with 24 features


,ID,occ_1,occ_2,occ_3,occ_4,occ_5
0,TFU2SNX1SE3X,27121210,32131011,32161416,19101010,23171541
1,KA65FZ2RIMC6,23171712,23171524,23171541,22101213,19111012
2,5RY27VMVWJMV,32121312,17151010,32161010,36111110,25111310
3,VL9V73PUWSXV,32131011,27121210,21131110,32161416,32131210
4,AJSPJ00VMPQY,14111117,23171524,27121210,32151013,19101010


In [6]:
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved:", SUBMISSION_PATH.resolve())
print("Submission shape:", submission.shape)

Saved: /content/submission.csv
Submission shape: (1196, 6)


## Next Steps

- **Improve Candidate Retrieval:**
- **Explore Deep Learning Approaches:**
- **Feature Engineering with SentenceTransformer Embeddings:**
    - Generate SentenceTransformer embeddings for `OCCUPATION_NAME`, `OCCUPATION_DESCRIPTION` from `occupations.csv`.
    - Generate SentenceTransformer embeddings for `NAME`, `DESCRIPTION` from `skills.csv`.
    - Create new features for the LGBMRanker by calculating cosine similarity or other distance metrics between skill embeddings and occupation embeddings, or by using the embeddings directly as features.

